# File Handling - JSON Files

JSON is a text format for structured data. It is used by web APIs, configuration files and data exchange.

| Function | Purpose |
|---|---|
| `json.dumps(obj)` | Python object -> JSON **string** |
| `json.loads(text)` | JSON string -> Python object |
| `json.dump(obj, file)` | Python object -> JSON **file** |
| `json.load(file)` | JSON file -> Python object |
| `indent=2` | Pretty printing |
| `sort_keys=True` | Sort dictionary keys |
| `ensure_ascii=False` | Keep non-ASCII characters readable |
| `default=func` | Convert objects JSON cannot handle |
| `json.JSONDecodeError` | Raised for invalid JSON |

The letter **s** in `dumps` / `loads` means **string**. Without it, the function works with a **file**.

---

## Type Mapping

| Python | JSON |
|---|---|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

### Important

* JSON has **no tuples**. A tuple becomes a list and stays a list after loading.
* JSON object keys are always **strings**. `{1: "a"}` becomes `{"1": "a"}`.
* Sets, dates and custom classes are **not** supported by default.

---

## Strings

```python
import json

text = json.dumps({"name": "Ann", "age": 30})
data = json.loads(text)
```

---

## Files

```python
with open("data.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2)

with open("data.json", "r", encoding="utf-8") as file:
    data = json.load(file)
```

Always pass `encoding="utf-8"`.

---

## Formatting Options

| Option | Effect |
|---|---|
| `indent=2` | Readable, multi-line output |
| `sort_keys=True` | Stable key order |
| `separators=(",", ":")` | Most compact output |
| `ensure_ascii=False` | Writes `é` instead of `\u00e9` |

---

## Unsupported Types

`json.dumps({1, 2})` raises `TypeError`. Give `default=` a function that converts the value:

```python
json.dumps(data, default=str)
```

---

## Invalid JSON

```python
try:
    json.loads("{'name': 'Ann'}")      # single quotes are not valid JSON
except json.JSONDecodeError as error:
    ...
```

`JSONDecodeError` is a subclass of `ValueError`.

## Source

https://docs.python.org/3/library/json.html

https://docs.python.org/3/tutorial/inputoutput.html#saving-structured-data-with-json

In [ ]:
import json
import tempfile
from datetime import date
from pathlib import Path

data = {"name": "Ann", "age": 30, "skills": ["python", "sql"], "active": True, "manager": None}

# Strings: dumps / loads
text = json.dumps(data)
print(text)
print(json.loads(text) == data)

# Formatting options
print(json.dumps(data, indent=2, sort_keys=True))
print(json.dumps({"a": 1, "b": [1, 2]}, separators=(",", ":")))
print(json.dumps({"city": "Café"}), json.dumps({"city": "Café"}, ensure_ascii=False))

# Files: dump / load
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.json"
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)
    with open(path, "r", encoding="utf-8") as file:
        loaded = json.load(file)
    print(loaded == data, path.read_text(encoding="utf-8").splitlines()[0])

# Type changes on a round trip
original = {"point": (1, 2), 1: "one"}
print(json.loads(json.dumps(original)))            # tuple -> list, key 1 -> "1"

# Unsupported types
try:
    json.dumps({"tags": {"a", "b"}})
except TypeError as error:
    print("TypeError:", error)

print(json.dumps({"day": date(2026, 9, 20)}, default=str))

def encode(value):
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"cannot serialize {type(value).__name__}")

print(json.dumps({"tags": {"b", "a"}}, default=encode))

# Invalid JSON
try:
    json.loads("{'name': 'Ann'}")
except json.JSONDecodeError as error:
    print("JSONDecodeError:", error.msg, "| is a ValueError:", isinstance(error, ValueError))